# Week 8: NLP Foundations & Text Embeddings
## Notebook 08: BERT Masked Language Modeling (MLM) Objective

### 1. Conceptual Intuition & Theoretical Background
Standard autoregressive language models (like GPT) are unidirectional: token $x_t$ can only attend to previous tokens $x_{<t}$ using a causal lower-triangular mask.

**BERT (Bidirectional Encoder Representations from Transformers)** learns deep bidirectional representations by conditioning on both left and right context simultaneously using the **Masked Language Model (MLM / Cloze Task)**:
1. $15\%$ of input tokens are selected at random.
2. Of the selected tokens:
   - $80\%$ are replaced with the special token `[MASK]`.
   - $10\%$ are replaced with a random vocabulary token.
   - $10\%$ are kept unchanged.
3. The model predicts the original ground-truth identity via Cross-Entropy Loss:
$$\mathcal{L}_{\text{MLM}} = -\sum_{i \in \text{Masked}} \log P(w_i = y_i \mid \tilde{X})$$
$$P(w_i = k \mid \tilde{X}) = \frac{\exp(h_i^T e_k)}{\sum_{v \in V} \exp(h_i^T e_v)}$$

---

### 2. Scratch Python Implementation: Dynamic MLM Preprocessing

In [ ]:
import numpy as np

# Vocabulary Definition
vocab = {
    '[PAD]': 0, '[UNK]': 1, '[CLS]': 2, '[SEP]': 3, '[MASK]': 4,
    'the': 5, 'cat': 6, 'sat': 7, 'on': 8, 'mat': 9, 'dog': 10,
    'slept': 11, 'under': 12, 'tree': 13, 'quick': 14, 'fox': 15
}
inv_vocab = {v: k for k, v in vocab.items()}

def create_mlm_batch(token_ids, mask_prob=0.15):
    """
    Implements BERT 80/10/10 masking rule.
    """
    input_ids = token_ids.copy()
    labels = np.full(len(token_ids), -100) # -100 is ignored in loss
    
    # Candidate tokens: exclude special tokens [CLS], [SEP]
    candidate_indices = [i for i, tid in enumerate(token_ids) if tid not in [0, 2, 3]]
    n_to_mask = max(1, int(len(candidate_indices) * mask_prob))
    
    selected_indices = np.random.choice(candidate_indices, size=n_to_mask, replace=False)
    
    for idx in selected_indices:
        labels[idx] = token_ids[idx] # Store original target
        
        prob = np.random.rand()
        if prob < 0.80:
            # 80% of time: replace with [MASK]
            input_ids[idx] = vocab['[MASK]']
        elif prob < 0.90:
            # 10% of time: replace with random word
            random_word_id = np.random.randint(5, len(vocab))
            input_ids[idx] = random_word_id
        else:
            # 10% of time: keep unchanged
            pass
            
    return input_ids, labels

# Test Sequence: "[CLS] the quick fox sat on the mat [SEP]"
raw_tokens = ['[CLS]', 'the', 'quick', 'fox', 'sat', 'on', 'the', 'mat', '[SEP]']
token_ids = np.array([vocab[t] for t in raw_tokens])

np.random.seed(42)
masked_input, ground_truth_labels = create_mlm_batch(token_ids, mask_prob=0.25)

print("Original Tokens:      ", [inv_vocab[i] for i in token_ids])
print("Masked Input to BERT: ", [inv_vocab[i] for i in masked_input])
print("Prediction Targets:   ", [inv_vocab[i] if i != -100 else '_' for i in ground_truth_labels])

### 3. Step-by-Step Code Walkthrough
1. **80/10/10 Masking Strategy**: Prevents fine-tuning mismatch, as the `[MASK]` token never appears in downstream tasks.
2. **Loss Masking (\-100)**: Gradients are computed only on the masked tokens, speeding up backward propagation.
3. **Bidirectional Attention**: The self-attention matrix retains full all-to-all visibility.

---

### 4. Industry Applications & Trade-offs
- **Bi-Encoder Dense Retrieval:** Pretrained BERT encoders power sentence embeddings for vector search.
- **Trade-off:** BERT cannot generate autoregressive text token-by-token.

---

### 5. Top Interview Questions & Answers
**Q1: Why doesn't BERT replace 100% of masked tokens with the `[MASK]` symbol?**
> **Answer:** If `[MASK]` were used 100% of the time, the model would never learn robust contextual representations for real words, creating a discrepancy during fine-tuning where `[MASK]` never appears.

**Q2: What is the difference between MLM and Causal Language Modeling?**
> **Answer:** MLM sees all tokens bidirectionally and predicts missing tokens simultaneously. Causal LM sees only preceding tokens ($x_{<t}$) and predicts the next sequential token autoregressively.